# RiskWeave — Operational Weather Forecasting

This is a **separate notebook** from `15_weather_forecasting.ipynb`. It will prepare recent ERA5-compatible daily weather inputs for the four already-trained models:

- Rainfall: 1-day and 7-day
- Mean temperature: 1-day and 7-day

We will proceed step by step. This first version only inspects existing files and data. It does not download data, overwrite datasets, retrain models, or generate predictions.

**Important:** The incoming data must match the training data in variable meaning, units, daily aggregation, district mapping, and feature-generation rules—not just column names.


## Step 1 — Import libraries and define project paths

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_WEATHER_DIR = PROJECT_ROOT / "data" / "raw" / "weather"
PROCESSED_WEATHER_DIR = PROJECT_ROOT / "data" / "processed" / "weather"
WEATHER_MODEL_DIR = PROJECT_ROOT / "models" / "weather_forecasting"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "operational_weather"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw weather directory:", RAW_WEATHER_DIR)
print("Processed weather directory:", PROCESSED_WEATHER_DIR)
print("Model directory:", WEATHER_MODEL_DIR)
print("Operational output directory:", OUTPUT_DIR)

Project root: c:\Users\Nethmi\Documents\GitHub\RiskWeave
Raw weather directory: c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\raw\weather
Processed weather directory: c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\processed\weather
Model directory: c:\Users\Nethmi\Documents\GitHub\RiskWeave\models\weather_forecasting
Operational output directory: c:\Users\Nethmi\Documents\GitHub\RiskWeave\outputs\operational_weather


## Step 2 — Find existing weather data files

This cell only lists CSV and Parquet files. It does not load or modify them.

In [4]:
# Load the original combined daily ERA5 dataset

ERA5_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "weather_era5_2006_2025"
    / "weather_daily_era5_2006-12-23_to_2025-12-19.csv"
)

if not ERA5_FILE.exists():
    raise FileNotFoundError(f"Dataset not found: {ERA5_FILE}")

weather_df = pd.read_csv(ERA5_FILE)

print("Dataset loaded successfully")
print("-" * 50)

print("Shape:", weather_df.shape)

print("\nColumn names:")
print(weather_df.columns.tolist())

print("\nFirst 5 records:")
display(weather_df.head())

print("\nData types:")
display(weather_df.dtypes.astype(str).to_frame("Data Type"))

Dataset loaded successfully
--------------------------------------------------
Shape: (173425, 25)

Column names:
['district', 'date', 'precipitation_sum', 'rain_sum', 'precipitation_hours', 'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 'relative_humidity_2m_mean', 'relative_humidity_2m_max', 'relative_humidity_2m_min', 'dew_point_2m_mean', 'wind_speed_10m_mean', 'wind_speed_10m_max', 'wind_direction_10m_dominant', 'cloud_cover_mean', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration', 'requested_latitude', 'requested_longitude', 'era5_grid_latitude', 'era5_grid_longitude', 'era5_grid_elevation_m', 'model', 'timezone']

First 5 records:


,district,date,precipitation_sum,rain_sum,precipitation_hours,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,relative_humidity_2m_max,...,cloud_cover_mean,shortwave_radiation_sum,et0_fao_evapotranspiration,requested_latitude,requested_longitude,era5_grid_latitude,era5_grid_longitude,era5_grid_elevation_m,model,timezone
0,Ampara,2006-12-23,6.8,6.8,22.0,25.3,27.1,24.2,89,94,...,99,13.74,2.80,7.124905,81.685472,7.0,81.75,48.0,era5,Asia/Colombo
1,Ampara,2006-12-24,3.4,3.4,19.0,25.6,27.3,23.8,87,94,...,97,11.36,2.60,7.124905,81.685472,7.0,81.75,48.0,era5,Asia/Colombo
2,Ampara,2006-12-25,3.8,3.8,11.0,25.2,26.4,24.0,83,87,...,100,6.42,2.08,7.124905,81.685472,7.0,81.75,48.0,era5,Asia/Colombo
3,Ampara,2006-12-26,1.3,1.3,8.0,25.5,27.4,23.6,81,89,...,98,12.89,3.19,7.124905,81.685472,7.0,81.75,48.0,era5,Asia/Colombo
4,Ampara,2006-12-27,0.0,0.0,0.0,25.4,28.0,23.5,77,84,...,93,14.11,3.48,7.124905,81.685472,7.0,81.75,48.0,era5,Asia/Colombo



Data types:


,Data Type
district,object
date,object
precipitation_sum,float64
rain_sum,float64
precipitation_hours,float64
temperature_2m_mean,float64
temperature_2m_max,float64
temperature_2m_min,float64
relative_humidity_2m_mean,int64
relative_humidity_2m_max,int64


In [5]:
# Convert date column to datetime
weather_df["date"] = pd.to_datetime(
    weather_df["date"],
    errors="coerce"
)

# Basic date information
print("EARLIEST DATE:", weather_df["date"].min())
print("LATEST DATE:", weather_df["date"].max())

print("\nTOTAL RECORDS:", len(weather_df))

# District coverage
print("\nNUMBER OF DISTRICTS:", weather_df["district"].nunique())

print("\nDISTRICTS:")
print(sorted(weather_df["district"].unique()))

# Records per district
district_counts = (
    weather_df.groupby("district")
    .size()
    .sort_values()
)

print("\nRECORDS PER DISTRICT:")
display(district_counts.to_frame("record_count"))

# Latest available date for each district
latest_dates = (
    weather_df.groupby("district")["date"]
    .max()
    .sort_values()
)

print("\nLATEST DATE PER DISTRICT:")
display(latest_dates.to_frame("latest_date"))

EARLIEST DATE: 2006-12-23 00:00:00
LATEST DATE: 2025-12-19 00:00:00

TOTAL RECORDS: 173425

NUMBER OF DISTRICTS: 25

DISTRICTS:
['Ampara', 'Anuradhapura', 'Badulla', 'Batticaloa', 'Colombo', 'Galle', 'Gampaha', 'Hambantota', 'Jaffna', 'Kalutara', 'Kandy', 'Kegalle', 'Kilinochchi', 'Kurunegala', 'Mannar', 'Matale', 'Matara', 'Moneragala', 'Mullaitivu', 'Nuwara Eliya', 'Polonnaruwa', 'Puttalam', 'Ratnapura', 'Trincomalee', 'Vavuniya']

RECORDS PER DISTRICT:


,record_count
district,
Ampara,6937
Anuradhapura,6937
Badulla,6937
Batticaloa,6937
Colombo,6937
Galle,6937
Gampaha,6937
Hambantota,6937
Jaffna,6937



LATEST DATE PER DISTRICT:


,latest_date
district,
Ampara,2025-12-19
Ratnapura,2025-12-19
Puttalam,2025-12-19
Polonnaruwa,2025-12-19
Nuwara Eliya,2025-12-19
Mullaitivu,2025-12-19
Moneragala,2025-12-19
Matara,2025-12-19
Matale,2025-12-19


In [6]:
# Check duplicate district-date combinations

duplicate_mask = weather_df.duplicated(
    subset=["district", "date"],
    keep=False
)

duplicates = weather_df.loc[
    duplicate_mask
].sort_values(["district", "date"])

print("DUPLICATE DISTRICT-DATE RECORDS:", len(duplicates))

if len(duplicates) > 0:
    display(duplicates.head(20))
else:
    print("No duplicate district-date records found.")

# Check missing dates within each district
missing_date_results = []

for district, group in weather_df.groupby("district"):

    district_dates = (
        group["date"]
        .dropna()
        .sort_values()
    )

    expected_dates = pd.date_range(
        start=district_dates.min(),
        end=district_dates.max(),
        freq="D"
    )

    missing_dates = expected_dates.difference(
        district_dates
    )

    missing_date_results.append({
        "district": district,
        "expected_days": len(expected_dates),
        "actual_days": district_dates.nunique(),
        "missing_days": len(missing_dates)
    })

missing_date_df = pd.DataFrame(missing_date_results)

print("\nMISSING DATE SUMMARY:")
display(missing_date_df)

print(
    "\nTOTAL MISSING DISTRICT-DATE RECORDS:",
    missing_date_df["missing_days"].sum()
)

DUPLICATE DISTRICT-DATE RECORDS: 0
No duplicate district-date records found.

MISSING DATE SUMMARY:


,district,expected_days,actual_days,missing_days
0,Ampara,6937,6937,0
1,Anuradhapura,6937,6937,0
2,Badulla,6937,6937,0
3,Batticaloa,6937,6937,0
4,Colombo,6937,6937,0
5,Galle,6937,6937,0
6,Gampaha,6937,6937,0
7,Hambantota,6937,6937,0
8,Jaffna,6937,6937,0
9,Kalutara,6937,6937,0



TOTAL MISSING DISTRICT-DATE RECORDS: 0


In [7]:
# Load existing district coordinates

COORDINATES_FILE = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "district_coordinates.csv"
)

if not COORDINATES_FILE.exists():
    raise FileNotFoundError(
        f"District coordinates not found: {COORDINATES_FILE}"
    )

coordinates = pd.read_csv(COORDINATES_FILE)

print("Coordinate dataset shape:", coordinates.shape)
print("\nColumns:", coordinates.columns.tolist())

display(coordinates.head(10))

print("\nNumber of districts:", coordinates["district"].nunique())

Coordinate dataset shape: (25, 6)

Columns: ['district', 'latitude', 'longitude', 'coordinate_method', 'coordinate_crs', 'boundary_source']


,district,latitude,longitude,coordinate_method,coordinate_crs,boundary_source
0,Ampara,7.124905,81.685472,representative_point,EPSG:4326,Sri Lanka NSDI
1,Anuradhapura,8.369011,80.440160,representative_point,EPSG:4326,Sri Lanka NSDI
2,Badulla,7.082047,81.088693,representative_point,EPSG:4326,Sri Lanka NSDI
3,Batticaloa,7.834330,81.409875,representative_point,EPSG:4326,Sri Lanka NSDI
4,Colombo,6.848546,80.028546,representative_point,EPSG:4326,Sri Lanka NSDI
5,Galle,6.203616,80.231034,representative_point,EPSG:4326,Sri Lanka NSDI
6,Gampaha,7.118525,80.021192,representative_point,EPSG:4326,Sri Lanka NSDI
7,Hambantota,6.276106,81.178027,representative_point,EPSG:4326,Sri Lanka NSDI
8,Jaffna,9.642790,79.908941,representative_point,EPSG:4326,Sri Lanka NSDI
9,Kalutara,6.575560,80.134539,representative_point,EPSG:4326,Sri Lanka NSDI



Number of districts: 25


In [8]:
import requests
from datetime import date, timedelta

# Select one district for API testing
test_district = coordinates.loc[
    coordinates["district"] == "Ampara"
].iloc[0]

# API configuration
API_URL = "https://archive-api.open-meteo.com/v1/archive"

# Continue after the last historical date
START_DATE = (
    weather_df["date"].max() + pd.Timedelta(days=1)
).strftime("%Y-%m-%d")

# Allow a 7-day buffer for ERA5 publication delay
END_DATE = (date.today() - timedelta(days=7)).isoformat()

DAILY_VARIABLES = [
    "precipitation_sum",
    "precipitation_hours",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "relative_humidity_2m_mean",
    "relative_humidity_2m_max",
    "relative_humidity_2m_min",
    "dew_point_2m_mean",
    "wind_speed_10m_mean",
    "wind_speed_10m_max",
]

params = {
    "latitude": float(test_district["latitude"]),
    "longitude": float(test_district["longitude"]),
    "start_date": START_DATE,
    "end_date": END_DATE,
    "daily": ",".join(DAILY_VARIABLES),
    "timezone": "Asia/Colombo",
    "models": "era5",
}

print("Testing district:", test_district["district"])
print("Requested date range:", START_DATE, "to", END_DATE)

response = requests.get(
    API_URL,
    params=params,
    timeout=60
)

print("\nHTTP status:", response.status_code)

response.raise_for_status()

api_data = response.json()

if "daily" not in api_data:
    raise ValueError(
        "The API response does not contain daily weather data."
    )

api_test_df = pd.DataFrame(api_data["daily"])

print("\nAPI test successful")
print("Returned records:", len(api_test_df))
print("Returned columns:", api_test_df.columns.tolist())

display(api_test_df.head())
display(api_test_df.tail())

Testing district: Ampara
Requested date range: 2025-12-20 to 2026-11-02

HTTP status: 200

API test successful
Returned records: 318
Returned columns: ['time', 'precipitation_sum', 'precipitation_hours', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration', 'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 'relative_humidity_2m_mean', 'relative_humidity_2m_max', 'relative_humidity_2m_min', 'dew_point_2m_mean', 'wind_speed_10m_mean', 'wind_speed_10m_max']


,time,precipitation_sum,precipitation_hours,shortwave_radiation_sum,et0_fao_evapotranspiration,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,relative_humidity_2m_max,relative_humidity_2m_min,dew_point_2m_mean,wind_speed_10m_mean,wind_speed_10m_max
0,2025-12-20,4.9,19.0,14.38,2.93,25.2,26.9,24.0,87,93,82,23.0,15.6,18.8
1,2025-12-21,2.1,14.0,12.29,2.85,25.2,26.8,23.5,84,91,77,22.4,16.4,20.6
2,2025-12-22,1.5,8.0,15.06,3.51,25.9,28.3,24.2,81,88,73,22.4,17.9,22.1
3,2025-12-23,1.6,9.0,18.68,3.80,25.7,28.2,23.3,84,92,76,22.8,15.6,19.8
4,2025-12-24,2.5,18.0,16.61,3.43,25.4,28.0,23.5,84,92,75,22.6,12.8,16.1


,time,precipitation_sum,precipitation_hours,shortwave_radiation_sum,et0_fao_evapotranspiration,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,relative_humidity_2m_max,relative_humidity_2m_min,dew_point_2m_mean,wind_speed_10m_mean,wind_speed_10m_max
313,2026-10-29,1.6,4.0,18.99,4.26,27.7,30.9,25.4,81,88,69,24.2,14.5,19.1
314,2026-10-30,1.2,4.0,20.21,4.35,28.0,32.1,25.8,80,89,60,24.0,5.8,13.8
315,2026-10-31,1.8,13.0,18.50,3.90,28.1,30.7,25.8,80,91,70,24.3,5.7,10.9
316,2026-11-01,0.6,6.0,22.37,4.77,27.8,31.2,25.2,78,90,59,23.4,6.4,11.0
317,2026-11-02,9.0,14.0,18.69,3.97,27.1,30.8,25.0,82,94,63,23.6,7.8,12.6


In [9]:
# Test the complete original ERA5 daily variable set for Ampara

FULL_DAILY_VARIABLES = [
    "precipitation_sum",
    "rain_sum",
    "precipitation_hours",
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "relative_humidity_2m_mean",
    "relative_humidity_2m_max",
    "relative_humidity_2m_min",
    "dew_point_2m_mean",
    "wind_speed_10m_mean",
    "wind_speed_10m_max",
    "wind_direction_10m_dominant",
    "cloud_cover_mean",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
]

params["daily"] = ",".join(FULL_DAILY_VARIABLES)

response = requests.get(
    API_URL,
    params=params,
    timeout=60
)

response.raise_for_status()

full_api_data = response.json()

full_api_test_df = pd.DataFrame(
    full_api_data["daily"]
)

print("Returned columns:")
print(full_api_test_df.columns.tolist())

print("\nNumber of returned records:", len(full_api_test_df))

print("\nMissing requested variables:")
print([
    col for col in FULL_DAILY_VARIABLES
    if col not in full_api_test_df.columns
])

display(full_api_test_df.head())

Returned columns:
['time', 'precipitation_sum', 'rain_sum', 'precipitation_hours', 'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 'relative_humidity_2m_mean', 'relative_humidity_2m_max', 'relative_humidity_2m_min', 'dew_point_2m_mean', 'wind_speed_10m_mean', 'wind_speed_10m_max', 'wind_direction_10m_dominant', 'cloud_cover_mean', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration']

Number of returned records: 318

Missing requested variables:
[]


,time,precipitation_sum,rain_sum,precipitation_hours,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,relative_humidity_2m_max,relative_humidity_2m_min,dew_point_2m_mean,wind_speed_10m_mean,wind_speed_10m_max,wind_direction_10m_dominant,cloud_cover_mean,shortwave_radiation_sum,et0_fao_evapotranspiration
0,2025-12-20,4.9,4.9,19.0,25.2,26.9,24.0,87,93,82,23.0,15.6,18.8,6,64,14.38,2.93
1,2025-12-21,2.1,2.1,14.0,25.2,26.8,23.5,84,91,77,22.4,16.4,20.6,357,83,12.29,2.85
2,2025-12-22,1.5,1.5,8.0,25.9,28.3,24.2,81,88,73,22.4,17.9,22.1,2,87,15.06,3.51
3,2025-12-23,1.6,1.6,9.0,25.7,28.2,23.3,84,92,76,22.8,15.6,19.8,3,73,18.68,3.80
4,2025-12-24,2.5,2.5,18.0,25.4,28.0,23.5,84,92,75,22.6,12.8,16.1,7,52,16.61,3.43


In [10]:
import time

# Operational ERA5 collection settings

NEW_WEATHER_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "weather_era5_operational"
)

NEW_WEATHER_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2025-12-20"

API_URL = "https://archive-api.open-meteo.com/v1/archive"

FULL_DAILY_VARIABLES = [
    "precipitation_sum",
    "rain_sum",
    "precipitation_hours",
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "relative_humidity_2m_mean",
    "relative_humidity_2m_max",
    "relative_humidity_2m_min",
    "dew_point_2m_mean",
    "wind_speed_10m_mean",
    "wind_speed_10m_max",
    "wind_direction_10m_dominant",
    "cloud_cover_mean",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
]

# Use the same 7-day publication buffer as the API test
END_DATE = (
    date.today() - timedelta(days=7)
).isoformat()

print("Collection period:", START_DATE, "to", END_DATE)
print("Number of districts:", len(coordinates))


def collect_operational_district(row):

    district = row["district"]

    params = {
        "latitude": float(row["latitude"]),
        "longitude": float(row["longitude"]),
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ",".join(FULL_DAILY_VARIABLES),
        "timezone": "Asia/Colombo",
        "models": "era5",
    }

    response = requests.get(
        API_URL,
        params=params,
        timeout=90
    )

    response.raise_for_status()

    result = response.json()

    if "daily" not in result:
        raise ValueError(
            f"No daily data returned for {district}"
        )

    district_df = pd.DataFrame(result["daily"])

    district_df = district_df.rename(
        columns={"time": "date"}
    )

    district_df.insert(0, "district", district)

    district_df["requested_latitude"] = float(row["latitude"])
    district_df["requested_longitude"] = float(row["longitude"])

    district_df["era5_grid_latitude"] = result.get(
        "latitude"
    )
    district_df["era5_grid_longitude"] = result.get(
        "longitude"
    )
    district_df["era5_grid_elevation_m"] = result.get(
        "elevation"
    )

    district_df["model"] = "era5"
    district_df["timezone"] = "Asia/Colombo"

    district_df["date"] = pd.to_datetime(
        district_df["date"]
    )

    return district_df


# Collect all districts
operational_district_data = []

for index, row in coordinates.iterrows():

    district = row["district"]

    try:
        print(
            f"[{index + 1}/{len(coordinates)}] "
            f"Collecting {district}..."
        )

        district_df = collect_operational_district(row)

        operational_district_data.append(district_df)

        print(
            f"  Records collected: {len(district_df)}"
        )

        time.sleep(0.5)

    except Exception as error:
        print(f"  ERROR for {district}: {error}")


# Combine successful district downloads
if operational_district_data:

    operational_weather_df = pd.concat(
        operational_district_data,
        ignore_index=True
    )

    output_file = (
        NEW_WEATHER_DIR
        / "operational_weather_era5.csv"
    )

    operational_weather_df.to_csv(
        output_file,
        index=False
    )

    print("\nCollection completed.")
    print("Successful districts:", len(operational_district_data))
    print("Total records:", len(operational_weather_df))
    print("Saved to:", output_file)

else:
    print("No district data was collected.")

Collection period: 2025-12-20 to 2026-11-02
Number of districts: 25
[1/25] Collecting Ampara...
  Records collected: 318
[2/25] Collecting Anuradhapura...
  Records collected: 318
[3/25] Collecting Badulla...
  Records collected: 318
[4/25] Collecting Batticaloa...
  Records collected: 318
[5/25] Collecting Colombo...
  Records collected: 318
[6/25] Collecting Galle...
  Records collected: 318
[7/25] Collecting Gampaha...
  Records collected: 318
[8/25] Collecting Hambantota...
  Records collected: 318
[9/25] Collecting Jaffna...
  Records collected: 318
[10/25] Collecting Kalutara...
  Records collected: 318
[11/25] Collecting Kandy...
  Records collected: 318
[12/25] Collecting Kegalle...
  Records collected: 318
[13/25] Collecting Kilinochchi...
  Records collected: 318
[14/25] Collecting Kurunegala...
  Records collected: 318
[15/25] Collecting Mannar...
  Records collected: 318
[16/25] Collecting Matale...
  Records collected: 318
[17/25] Collecting Matara...
  Records collected: 

In [11]:
# Combine historical and operational ERA5 datasets

historical_df = pd.read_csv(ERA5_FILE)

historical_df["date"] = pd.to_datetime(
    historical_df["date"]
)

operational_weather_df["date"] = pd.to_datetime(
    operational_weather_df["date"]
)

# Match the historical dataset's column order
combined_weather_df = pd.concat(
    [
        historical_df,
        operational_weather_df
    ],
    ignore_index=True
)

# Sort by district and date
combined_weather_df = combined_weather_df.sort_values(
    ["district", "date"]
).reset_index(drop=True)

print("Historical records:", len(historical_df))
print("New operational records:", len(operational_weather_df))
print("Combined records:", len(combined_weather_df))

print("\nCombined date range:")
print(
    combined_weather_df["date"].min(),
    "to",
    combined_weather_df["date"].max()
)

print("\nCombined dataset shape:")
print(combined_weather_df.shape)

display(combined_weather_df.tail())

Historical records: 173425
New operational records: 7950
Combined records: 181375

Combined date range:
2006-12-23 00:00:00 to 2026-11-02 00:00:00

Combined dataset shape:
(181375, 25)


,district,date,precipitation_sum,rain_sum,precipitation_hours,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,relative_humidity_2m_max,...,cloud_cover_mean,shortwave_radiation_sum,et0_fao_evapotranspiration,requested_latitude,requested_longitude,era5_grid_latitude,era5_grid_longitude,era5_grid_elevation_m,model,timezone
181370,Vavuniya,2026-10-29,0.0,0.0,0.0,28.4,32.8,25.6,81,94,...,97,20.45,4.58,8.836117,80.486152,8.75,80.5,78.0,era5,Asia/Colombo
181371,Vavuniya,2026-10-30,0.5,0.5,4.0,28.2,32.6,25.1,85,95,...,43,20.60,4.36,8.836117,80.486152,8.75,80.5,78.0,era5,Asia/Colombo
181372,Vavuniya,2026-10-31,0.1,0.1,1.0,28.3,33.5,24.8,81,96,...,46,21.05,4.60,8.836117,80.486152,8.75,80.5,78.0,era5,Asia/Colombo
181373,Vavuniya,2026-11-01,0.0,0.0,0.0,27.9,32.4,24.4,80,97,...,76,18.81,3.99,8.836117,80.486152,8.75,80.5,78.0,era5,Asia/Colombo
181374,Vavuniya,2026-11-02,1.9,1.9,9.0,27.7,32.9,24.6,83,95,...,98,18.99,4.10,8.836117,80.486152,8.75,80.5,78.0,era5,Asia/Colombo


In [12]:
# Check duplicate records after combining datasets

duplicate_count = combined_weather_df.duplicated(
    subset=["district", "date"]
).sum()

print("Duplicate district-date records:", duplicate_count)

# Check new record counts by district
new_record_counts = (
    operational_weather_df
    .groupby("district")
    .size()
    .sort_values()
)

print("\nNew records per district:")
display(new_record_counts.to_frame("new_records"))

# Check latest date by district
latest_by_district = (
    combined_weather_df
    .groupby("district")["date"]
    .max()
)

print("\nLatest date by district:")
display(latest_by_district.to_frame("latest_date"))

# Confirm expected total
expected_records = len(historical_df) + len(operational_weather_df)

print("\nExpected records:", expected_records)
print("Actual records:", len(combined_weather_df))
print("Record count matches:", len(combined_weather_df) == expected_records)

Duplicate district-date records: 0

New records per district:


,new_records
district,
Ampara,318
Anuradhapura,318
Badulla,318
Batticaloa,318
Colombo,318
Galle,318
Gampaha,318
Hambantota,318
Jaffna,318



Latest date by district:


,latest_date
district,
Ampara,2026-11-02
Anuradhapura,2026-11-02
Badulla,2026-11-02
Batticaloa,2026-11-02
Colombo,2026-11-02
Galle,2026-11-02
Gampaha,2026-11-02
Hambantota,2026-11-02
Jaffna,2026-11-02



Expected records: 181375
Actual records: 181375
Record count matches: True


In [13]:
# Save the combined historical and operational weather dataset

COMBINED_WEATHER_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "weather_era5_combined_2006_2026.csv"
)

combined_weather_df.to_csv(
    COMBINED_WEATHER_FILE,
    index=False
)

print("Combined dataset saved successfully!")
print("File:", COMBINED_WEATHER_FILE)
print("Records:", len(combined_weather_df))
print("Columns:", len(combined_weather_df.columns))
print("Latest date:", combined_weather_df["date"].max())

Combined dataset saved successfully!
File: c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\processed\weather_era5_combined_2006_2026.csv
Records: 181375
Columns: 25
Latest date: 2026-11-02 00:00:00


In [14]:
from pathlib import Path
import joblib

MODEL_DIR = PROJECT_ROOT / "models" / "weather_forecasting"

model_files = [
    "precipitation_sum_1day_model.joblib",
    "precipitation_sum_7day_model.joblib",
    "temperature_2m_mean_1day_model.joblib",
    "temperature_2m_mean_7day_model.joblib",
]

for filename in model_files:
    model_path = MODEL_DIR / filename

    print("\n" + "=" * 65)
    print("Model:", filename)

    if not model_path.exists():
        print("FILE NOT FOUND:", model_path)
        continue

    artifact = joblib.load(model_path)

    print("Artifact type:", type(artifact).__name__)

    if isinstance(artifact, dict):
        print("Artifact keys:", list(artifact.keys()))

        if "feature_columns" in artifact:
            print("Expected feature count:", len(artifact["feature_columns"]))
            print("First 15 features:", artifact["feature_columns"][:15])

        if "model" in artifact:
            print("Stored model type:", type(artifact["model"]).__name__)

    elif hasattr(artifact, "n_features_in_"):
        print("Expected feature count:", artifact.n_features_in_)

        if hasattr(artifact, "feature_names_in_"):
            print("First 15 features:", artifact.feature_names_in_[:15])

    del artifact


Model: precipitation_sum_1day_model.joblib
Artifact type: dict
Artifact keys: ['model', 'preprocessor', 'target', 'horizon_days', 'model_name', 'feature_columns', 'categorical_features', 'numerical_features', 'test_metrics']
Expected feature count: 91
First 15 features: ['district', 'precipitation_sum_lag_1', 'precipitation_sum_lag_2', 'precipitation_sum_lag_3', 'precipitation_sum_lag_7', 'precipitation_sum_lag_14', 'precipitation_sum_lag_28', 'temperature_2m_mean_lag_1', 'temperature_2m_mean_lag_2', 'temperature_2m_mean_lag_3', 'temperature_2m_mean_lag_7', 'temperature_2m_mean_lag_14', 'temperature_2m_mean_lag_28', 'temperature_2m_max_lag_1', 'temperature_2m_max_lag_2']
Stored model type: XGBRegressor

Model: precipitation_sum_7day_model.joblib
Artifact type: dict
Artifact keys: ['model', 'preprocessor', 'target', 'horizon_days', 'model_name', 'feature_columns', 'categorical_features', 'numerical_features', 'test_metrics']
Expected feature count: 91
First 15 features: ['district', 'p

In [15]:
import json

TRAINING_NOTEBOOK = (
    PROJECT_ROOT / "notebooks" / "15_weather_forecasting.ipynb"
)

if not TRAINING_NOTEBOOK.exists():
    print("Training notebook not found:", TRAINING_NOTEBOOK)
else:
    with open(TRAINING_NOTEBOOK, "r", encoding="utf-8") as file:
        training_notebook = json.load(file)

    search_terms = [
        "MODEL_FEATURE_COLUMNS",
        "BASE_WEATHER_FEATURES",
        "lag_",
        "rolling",
        "feature_df",
    ]

    for cell_number, cell in enumerate(training_notebook["cells"]):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        if any(term in source for term in search_terms):
            print("\n" + "=" * 80)
            print(f"TRAINING NOTEBOOK CELL {cell_number}")
            print("=" * 80)
            print(source)


TRAINING NOTEBOOK CELL 15
# ============================================================
# CELL 16: CREATE HISTORICAL WEATHER FEATURES
# ============================================================

# Sort observations by district and date
weather_sorted = weather_clean.sort_values(
    ["district", "date"]
).reset_index(drop=True)

# Historical weather variables used to create lag features
BASE_WEATHER_FEATURES = [
    "precipitation_sum",
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "relative_humidity_2m_mean",
    "relative_humidity_2m_max",
    "relative_humidity_2m_min",
    "dew_point_2m_mean",
    "wind_speed_10m_mean",
    "wind_speed_10m_max",
    "cloud_cover_mean",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration"
]

# Verify that all selected variables exist
missing_features = [
    col for col in BASE_WEATHER_FEATURES
    if col not in weather_sorted.columns
]

if missing_features:
    raise ValueError(
        f"Missing

In [16]:
import numpy as np
import pandas as pd
import gc

# Use the combined dataset
weather_sorted = combined_weather_df.copy()

weather_sorted["date"] = pd.to_datetime(
    weather_sorted["date"]
)

weather_sorted = weather_sorted.sort_values(
    ["district", "date"]
).reset_index(drop=True)

# Exact variables used during model training
BASE_WEATHER_FEATURES = [
    "precipitation_sum",
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "relative_humidity_2m_mean",
    "relative_humidity_2m_max",
    "relative_humidity_2m_min",
    "dew_point_2m_mean",
    "wind_speed_10m_mean",
    "wind_speed_10m_max",
    "cloud_cover_mean",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration"
]

LAG_DAYS = [1, 2, 3, 7, 14, 28]

ROLLING_VARIABLES = [
    "precipitation_sum",
    "temperature_2m_mean"
]

ROLLING_WINDOWS = [7, 14, 30]

# Create feature dataframe
feature_df = weather_sorted[["district", "date"]].copy()

# Lag features
for col in BASE_WEATHER_FEATURES:
    for lag in LAG_DAYS:
        feature_df[f"{col}_lag_{lag}"] = (
            weather_sorted.groupby("district")[col]
            .shift(lag)
        )

# Rolling averages using previous observations only
for col in ROLLING_VARIABLES:
    for window in ROLLING_WINDOWS:
        feature_df[f"{col}_rolling_mean_{window}"] = (
            weather_sorted.groupby("district")[col]
            .transform(
                lambda s: s.shift(1).rolling(
                    window=window,
                    min_periods=window
                ).mean()
            )
        )

# Origin-date calendar features
feature_df["origin_dayofyear_sin"] = np.sin(
    2 * np.pi * feature_df["date"].dt.dayofyear / 365.25
)

feature_df["origin_dayofyear_cos"] = np.cos(
    2 * np.pi * feature_df["date"].dt.dayofyear / 365.25
)

feature_df["origin_dayofweek"] = (
    feature_df["date"].dt.dayofweek
)

FEATURE_COLUMNS = [
    col for col in feature_df.columns
    if col not in ["district", "date"]
]

print("Feature engineering completed.")
print("Feature dataset:", feature_df.shape)
print("Historical features:", len(FEATURE_COLUMNS))
print("Districts:", feature_df["district"].nunique())

Feature engineering completed.
Feature dataset: (181375, 89)
Historical features: 87
Districts: 25


In [17]:
import joblib

MODEL_DIR = PROJECT_ROOT / "models" / "weather_forecasting"

MODEL_FILES = {
    ("precipitation_sum", 1): "precipitation_sum_1day_model.joblib",
    ("precipitation_sum", 7): "precipitation_sum_7day_model.joblib",
    ("temperature_2m_mean", 1): "temperature_2m_mean_1day_model.joblib",
    ("temperature_2m_mean", 7): "temperature_2m_mean_7day_model.joblib"
}

model_artifacts = {}

for task, filename in MODEL_FILES.items():
    artifact = joblib.load(MODEL_DIR / filename)

    model_artifacts[task] = artifact

    print(
        task,
        "| Model:", artifact["model_name"],
        "| Expected features:", len(artifact["feature_columns"])
    )

    assert len(artifact["feature_columns"]) == 91

# Confirm the exact feature structure
first_artifact = next(iter(model_artifacts.values()))

MODEL_FEATURE_COLUMNS = first_artifact["feature_columns"]

for task, artifact in model_artifacts.items():
    assert artifact["feature_columns"] == MODEL_FEATURE_COLUMNS

print("\nAll four model feature configurations match.")

('precipitation_sum', 1) | Model: XGBoost | Expected features: 91
('precipitation_sum', 7) | Model: HistGradientBoosting | Expected features: 91
('temperature_2m_mean', 1) | Model: RandomForest | Expected features: 91
('temperature_2m_mean', 7) | Model: RandomForest | Expected features: 91

All four model feature configurations match.


In [20]:

# Select the latest available date for each district
# Memory-efficient approach: avoid sorting and copying the full feature_df

latest_indices = (
    feature_df.groupby("district")["date"].idxmax()
)

# Select only the 25 latest district rows
latest_rows = feature_df.loc[latest_indices].copy()

latest_rows = latest_rows.reset_index(drop=True)

# Add target-date calendar features separately for each horizon
def prepare_forecast_input(latest_data, horizon):

    forecast_input = latest_data.copy()

    forecast_input["target_date"] = (
        forecast_input["date"]
        + pd.to_timedelta(horizon, unit="D")
    )

    forecast_input["target_dayofyear_sin"] = np.sin(
        2 * np.pi
        * forecast_input["target_date"].dt.dayofyear / 365.25
    )

    forecast_input["target_dayofyear_cos"] = np.cos(
        2 * np.pi
        * forecast_input["target_date"].dt.dayofyear / 365.25
    )

    forecast_input["target_month"] = (
        forecast_input["target_date"].dt.month
    )

    return forecast_input


print("Latest origin date:", latest_rows["date"].max())
print("Districts:", latest_rows["district"].nunique())
print("Latest rows:", len(latest_rows))


Latest origin date: 2026-11-02 00:00:00
Districts: 25
Latest rows: 25


In [21]:
prediction_records = []

for (target, horizon), artifact in model_artifacts.items():

    print(f"Predicting {target} | {horizon}-day horizon")

    forecast_input = prepare_forecast_input(
        latest_rows,
        horizon
    )

    X = forecast_input[MODEL_FEATURE_COLUMNS].copy()

    numerical_features = artifact["numerical_features"]

    X[numerical_features] = X[numerical_features].astype(
        np.float32
    )

    if X.isna().any().any():
        missing = X.isna().sum()
        raise ValueError(
            f"Missing input features for {target}, {horizon} days:\n"
            f"{missing[missing > 0]}"
        )

    # Apply the original fitted preprocessor
    X_processed = artifact["preprocessor"].transform(X)

    X_processed = np.asarray(
        X_processed,
        dtype=np.float32
    )

    # Generate predictions
    predictions = artifact["model"].predict(X_processed)

    # Rainfall cannot be negative
    if target == "precipitation_sum":
        predictions = np.maximum(predictions, 0)

    for i, prediction in enumerate(predictions):
        prediction_records.append({
            "district": forecast_input.iloc[i]["district"],
            "forecast_origin_date": forecast_input.iloc[i]["date"],
            "target_date": forecast_input.iloc[i]["target_date"],
            "target": target,
            "horizon_days": horizon,
            "predicted_value": float(prediction),
            "model": artifact["model_name"]
        })

    del X, X_processed, predictions
    gc.collect()

predictions_df = pd.DataFrame(prediction_records)

print("\nForecast generation completed.")
print("Prediction records:", len(predictions_df))
display(predictions_df.head(10))

Predicting precipitation_sum | 1-day horizon
Predicting precipitation_sum | 7-day horizon
Predicting temperature_2m_mean | 1-day horizon
Predicting temperature_2m_mean | 7-day horizon

Forecast generation completed.
Prediction records: 100


,district,forecast_origin_date,target_date,target,horizon_days,predicted_value,model
0,Ampara,2026-11-02,2026-11-03,precipitation_sum,1,4.561803,XGBoost
1,Anuradhapura,2026-11-02,2026-11-03,precipitation_sum,1,5.999666,XGBoost
2,Badulla,2026-11-02,2026-11-03,precipitation_sum,1,10.414537,XGBoost
3,Batticaloa,2026-11-02,2026-11-03,precipitation_sum,1,4.453818,XGBoost
4,Colombo,2026-11-02,2026-11-03,precipitation_sum,1,8.162950,XGBoost
5,Galle,2026-11-02,2026-11-03,precipitation_sum,1,8.288243,XGBoost
6,Gampaha,2026-11-02,2026-11-03,precipitation_sum,1,7.493551,XGBoost
7,Hambantota,2026-11-02,2026-11-03,precipitation_sum,1,6.545065,XGBoost
8,Jaffna,2026-11-02,2026-11-03,precipitation_sum,1,4.187075,XGBoost
9,Kalutara,2026-11-02,2026-11-03,precipitation_sum,1,8.646180,XGBoost


In [22]:
PREDICTION_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "weather_forecasting"
    / "operational"
)

PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_FILE = (
    PREDICTION_DIR
    / "latest_weather_forecasts.csv"
)

predictions_df.to_csv(
    PREDICTION_FILE,
    index=False
)

print("Operational predictions saved.")
print("File:", PREDICTION_FILE)
print("Total predictions:", len(predictions_df))

Operational predictions saved.
File: c:\Users\Nethmi\Documents\GitHub\RiskWeave\outputs\weather_forecasting\operational\latest_weather_forecasts.csv
Total predictions: 100


In [23]:
print("FORECAST SUMMARY")
print("=" * 60)

print("Forecast origin dates:")
display(
    predictions_df.groupby("horizon_days")[
        "forecast_origin_date"
    ].agg(["min", "max"])
)

print("\nTarget dates:")
display(
    predictions_df.groupby("horizon_days")[
        "target_date"
    ].agg(["min", "max"])
)

print("\nPredictions per target and horizon:")
display(
    predictions_df.groupby(
        ["target", "horizon_days"]
    ).size().to_frame("prediction_count")
)

print("\nMissing predictions:", predictions_df["predicted_value"].isna().sum())
print("Negative rainfall predictions:",
      (predictions_df.loc[
          predictions_df["target"] == "precipitation_sum",
          "predicted_value"
      ] < 0).sum())

FORECAST SUMMARY
Forecast origin dates:


,min,max
horizon_days,,
1,2026-11-02,2026-11-02
7,2026-11-02,2026-11-02



Target dates:


,min,max
horizon_days,,
1,2026-11-03,2026-11-03
7,2026-11-09,2026-11-09



Predictions per target and horizon:


prediction_count
target              horizon_days                  
precipitation_sum   1                           25
                    7                           25
temperature_2m_mean 1                           25
                    7                           25


Missing predictions: 0
Negative rainfall predictions: 0


In [24]:
PREDICTION_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "weather_forecasting"
    / "operational"
)

PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_FILE = (
    PREDICTION_DIR / "latest_weather_forecasts.csv"
)

predictions_df.to_csv(PREDICTION_FILE, index=False)

print("Predictions saved successfully!")
print("Location:", PREDICTION_FILE)
print("Total predictions:", len(predictions_df))

Predictions saved successfully!
Location: c:\Users\Nethmi\Documents\GitHub\RiskWeave\outputs\weather_forecasting\operational\latest_weather_forecasts.csv
Total predictions: 100


In [25]:
import requests
import pandas as pd
from pathlib import Path

# Project paths
COORD_FILE = PROJECT_ROOT / "data" / "reference" / "district_coordinates.csv"

LIVE_DIR = PROJECT_ROOT / "data" / "raw" / "weather_live"
LIVE_DIR.mkdir(parents=True, exist_ok=True)

LIVE_WEATHER_FILE = LIVE_DIR / "live_weather_forecasts.csv"

# Load the existing 25 district coordinates
district_coords = pd.read_csv(COORD_FILE)

API_URL = "https://api.open-meteo.com/v1/forecast"

daily_variables = [
    "precipitation_sum",
    "rain_sum",
    "precipitation_hours",
    "temperature_2m_max",
    "temperature_2m_min",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
    "wind_speed_10m_max",
    "wind_direction_10m_dominant"
]

hourly_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "wind_speed_10m",
    "cloud_cover"
]

live_records = []
failed_districts = []

for _, district in district_coords.iterrows():

    district_name = district["district"]

    params = {
        "latitude": district["latitude"],
        "longitude": district["longitude"],
        "daily": ",".join(daily_variables),
        "hourly": ",".join(hourly_variables),
        "forecast_days": 8,
        "timezone": "Asia/Colombo"
    }

    try:
        response = requests.get(
            API_URL,
            params=params,
            timeout=60
        )
        response.raise_for_status()

        data = response.json()

        # Daily forecast data
        daily_df = pd.DataFrame(data["daily"])
        daily_df["date"] = pd.to_datetime(daily_df["time"])

        # Hourly forecast data
        hourly_df = pd.DataFrame(data["hourly"])
        hourly_df["date"] = pd.to_datetime(
            hourly_df["time"]
        ).dt.normalize()

        # Calculate daily summaries from hourly forecasts
        hourly_summary = (
            hourly_df.groupby("date")
            .agg(
                temperature_2m_mean=("temperature_2m", "mean"),
                relative_humidity_2m_mean=("relative_humidity_2m", "mean"),
                relative_humidity_2m_max=("relative_humidity_2m", "max"),
                relative_humidity_2m_min=("relative_humidity_2m", "min"),
                dew_point_2m_mean=("dew_point_2m", "mean"),
                wind_speed_10m_mean=("wind_speed_10m", "mean"),
                cloud_cover_mean=("cloud_cover", "mean")
            )
            .reset_index()
        )

        district_df = daily_df.merge(
            hourly_summary,
            on="date",
            how="left"
        )

        district_df["district"] = district_name
        district_df["data_source"] = "Open-Meteo Forecast API"
        district_df["retrieved_at"] = pd.Timestamp.now(
            tz="Asia/Colombo"
        ).strftime("%Y-%m-%d %H:%M:%S")

        live_records.append(district_df)

        print(f"Collected: {district_name}")

    except Exception as error:
        failed_districts.append({
            "district": district_name,
            "error": str(error)
        })
        print(f"Failed: {district_name} - {error}")

# Combine successful district forecasts
if live_records:
    live_weather_df = pd.concat(
        live_records,
        ignore_index=True
    )

    live_weather_df = live_weather_df.sort_values(
        ["district", "date"]
    ).reset_index(drop=True)

    live_weather_df.to_csv(
        LIVE_WEATHER_FILE,
        index=False
    )

    print("\nLIVE WEATHER COLLECTION SUMMARY")
    print("Districts collected:", live_weather_df["district"].nunique())
    print("Forecast records:", len(live_weather_df))
    print("Forecast dates:", live_weather_df["date"].min(),
          "to", live_weather_df["date"].max())
    print("Saved:", LIVE_WEATHER_FILE)

else:
    print("No district forecast data was collected.")

if failed_districts:
    print("\nFailed districts:")
    display(pd.DataFrame(failed_districts))

Collected: Ampara
Collected: Anuradhapura
Collected: Badulla
Collected: Batticaloa
Collected: Colombo
Collected: Galle
Collected: Gampaha
Collected: Hambantota
Collected: Jaffna
Collected: Kalutara
Collected: Kandy
Collected: Kegalle
Collected: Kilinochchi
Collected: Kurunegala
Collected: Mannar
Collected: Matale
Collected: Matara
Collected: Moneragala
Collected: Mullaitivu
Collected: Nuwara Eliya
Collected: Polonnaruwa
Collected: Puttalam
Collected: Ratnapura
Collected: Trincomalee
Collected: Vavuniya

LIVE WEATHER COLLECTION SUMMARY
Districts collected: 25
Forecast records: 200
Forecast dates: 2026-11-09 00:00:00 to 2026-11-16 00:00:00
Saved: c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\raw\weather_live\live_weather_forecasts.csv


In [26]:
# Check actual dates returned by the API
today = pd.Timestamp.now(
    tz="Asia/Colombo"
).date()

print("Today's date:", today)

print("Earliest API forecast date:",
      live_weather_df["date"].min().date())

print("Latest API forecast date:",
      live_weather_df["date"].max().date())

print("\nAvailable forecast dates:")

display(
    live_weather_df[
        ["date"]
    ]
    .drop_duplicates()
    .sort_values("date")
    .reset_index(drop=True)
)

print("\nTomorrow:",
      today + pd.Timedelta(days=1))

print("Seven days from today:",
      today + pd.Timedelta(days=7))

Today's date: 2026-11-09
Earliest API forecast date: 2026-11-09
Latest API forecast date: 2026-11-16

Available forecast dates:


,date
0,2026-11-09
1,2026-11-10
2,2026-11-11
3,2026-11-12
4,2026-11-13
5,2026-11-14
6,2026-11-15
7,2026-11-16



Tomorrow: 2026-11-10
Seven days from today: 2026-11-16


In [27]:

# Prepare dashboard weather forecasts

today = pd.Timestamp.now(
    tz="Asia/Colombo"
).date()

seven_days_ahead = today + pd.Timedelta(days=7)

# Include today through 7 days ahead
dashboard_weather_df = live_weather_df[
    (live_weather_df["date"].dt.date >= today) &
    (live_weather_df["date"].dt.date <= seven_days_ahead)
].copy()

dashboard_weather_df = dashboard_weather_df.sort_values(
    ["date", "district"]
).reset_index(drop=True)

# Save dashboard-ready forecasts
DASHBOARD_WEATHER_FILE = (
    LIVE_DIR / "dashboard_weather_forecasts.csv"
)

dashboard_weather_df.to_csv(
    DASHBOARD_WEATHER_FILE,
    index=False
)

print("DASHBOARD WEATHER DATA")
print("=" * 50)
print("Forecast period:", today, "to", seven_days_ahead)
print("Number of dates:", dashboard_weather_df["date"].nunique())
print("Districts:", dashboard_weather_df["district"].nunique())
print("Records:", len(dashboard_weather_df))
print("Saved:", DASHBOARD_WEATHER_FILE)

display(
    dashboard_weather_df[
        [
            "district",
            "date",
            "precipitation_sum",
            "temperature_2m_mean",
            "temperature_2m_max",
            "temperature_2m_min"
        ]
    ].head(10)
)


DASHBOARD WEATHER DATA
Forecast period: 2026-11-09 to 2026-11-16
Number of dates: 8
Districts: 25
Records: 200
Saved: c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\raw\weather_live\dashboard_weather_forecasts.csv


,district,date,precipitation_sum,temperature_2m_mean,temperature_2m_max,temperature_2m_min
0,Ampara,2026-11-09,10.1,25.791667,30.2,23.2
1,Anuradhapura,2026-11-09,29.5,25.058333,28.8,23.4
2,Badulla,2026-11-09,12.2,21.700000,26.6,19.5
3,Batticaloa,2026-11-09,7.7,25.583333,29.1,23.5
4,Colombo,2026-11-09,56.4,25.520833,30.5,23.3
5,Galle,2026-11-09,19.0,24.966667,31.4,22.8
6,Gampaha,2026-11-09,23.6,25.333333,30.5,23.5
7,Hambantota,2026-11-09,7.5,26.133333,30.6,23.9
8,Jaffna,2026-11-09,20.4,25.912500,29.0,23.9
9,Kalutara,2026-11-09,31.3,24.362500,30.6,22.3
